# 6. Real Ethernet ON/OFF cycle

## Before you start

Use [the Ethernet guide](README.md) and the project's Python 3.12 kernel. Each
notebook runs independently against **shutter**, CH **0**, serial **68800777**,
firmware **1.0.7**, KEH3 middle bay at **192.168.1.19:40305**. Port **40303** is
hub management. Keep the hub powered, Ethernet connected and USB IN disconnected.
Close competing owners before connecting. No simulator, scan or USB fallback is used.

Construction is passive. The context releases communication on success or error;
it sends no output command. If release fails, retain `hub` and resolve the reported
error before retrying `hub.close()`. Restart the kernel after source changes.

In [2]:
import sys
from dataclasses import asdict
from pathlib import Path
from pprint import pprint

sys.path.insert(0, str(Path("docs/examples") if Path("docs/examples").is_dir() else Path.cwd()))
from shutter_config import SHUTTER  # noqa: E402

from thorlabs_shutter_control import ShutterHub  # noqa: E402

print(SHUTTER.name, SHUTTER.route.endpoint)

shutter 192.168.1.19:40305


**This cell actuates the connected SH05R once.** Use the independently blocked
source, isolated trigger, intact key/interlock and exact setup approved for this
session. Confirm Manual mode first using notebook5 if necessary. Source protection
must remain in place; electrical feedback does not independently measure the blade.

The sequence is OFF/readback, the configured2s OFF pause, one1s ON hold, then
priority OFF/readback. `pulse` uses the same ON/OFF driver and preserves command
and cleanup errors. It is a host-timed hold, not calibrated optical exposure.
No simulator, SDK, extra bay or automatic ON retry is used. Final output is OFF;
context exit releases communication. Keep `hub` if release fails.

The direct calls on a connected owner are `hub.set_solenoid(name, True)` for ON
and `hub.set_solenoid(name, False)` for OFF. They do not select a mode implicitly.
See [the API](../python-api.md) and [wire commands](../INTERFACE.md).

In [3]:
import time

with ShutterHub([SHUTTER]) as hub:
    hub.connect(SHUTTER.name)
    hub.safe_state(SHUTTER.name)
    time.sleep(SHUTTER.policy.min_off_s)
    status = hub.pulse(SHUTTER.name, 1.0)
    pprint(asdict(status.feedback))

{'channel_enabled': None,
 'electrical_observed_at': '2026-11-06T21:26:57.983524+00:00',
 'electrical_state': 'off',
 'interlock_enabled': True,
 'key_enabled': None,
 'operating_mode': 'manual',
 'operating_state': 'unknown',
 'shutter_state': 'unknown',
 'status_bits': 8192}
